## **Food Delivery Time Prediction Using AdaBoost Regressor with Hyperparameter Tuning**

### **Problem Statement**
Food delivery platforms need to accurately predict delivery time to improve
customer satisfaction and operational efficiency. Delivery time is influenced
by multiple real-world factors such as distance, traffic level, weather
conditions, preparation time, vehicle type and partner ratings.

This project uses AdaBoost Regressor — a powerful sequential boosting
ensemble method — to predict food delivery time (in minutes) on a dataset
of 1000 records. Unlike Random Forest which builds trees in parallel,
AdaBoost builds trees sequentially, each focusing on the errors made by
the previous model — resulting in a stronger and more accurate predictor.

### **Objective**
- Predict food delivery time (in minutes) using AdaBoost Regressor

- Perform Feature Engineering to extract 4 new meaningful features
- Apply One-Hot Encoding on categorical features
- Follow correct ML pipeline — Encoding → Split → Model
- Improve model performance using GridSearchCV Hyperparameter Tuning
- Evaluate using MAE, MSE, RMSE and R² Score
- Compare Before and After tuning performance

### 1. Import Libraries

In [74]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 2. Load Dataset

In [75]:
df = pd.read_csv("food_delivery_dataset.csv")

### 3. Data Understanding

In [76]:
df.head()

,Order_ID,Distance_km,Order_Time,Traffic_Level,Weather,Restaurant_Rating,Prep_Time_min,Delivery_Partner_Rating,Vehicle_Type,Delivery_Time_min
0,1001,4.06,21,Medium,Rainy,4.6,20,4.7,Bike,58
1,1002,9.53,17,High,Clear,3.4,25,4.3,Scooter,90
2,1003,7.45,12,Medium,Cloudy,3.7,26,3.9,Bike,78
3,1004,6.19,17,Low,Rainy,3.7,28,4.5,Bike,69
4,1005,1.98,14,Medium,Clear,5.0,27,4.4,Scooter,46


In [77]:
df.shape

(1000, 10)

In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Order_ID                 1000 non-null   int64  
 1   Distance_km              1000 non-null   float64
 2   Order_Time               1000 non-null   int64  
 3   Traffic_Level            1000 non-null   object 
 4   Weather                  1000 non-null   object 
 5   Restaurant_Rating        1000 non-null   float64
 6   Prep_Time_min            1000 non-null   int64  
 7   Delivery_Partner_Rating  1000 non-null   float64
 8   Vehicle_Type             1000 non-null   object 
 9   Delivery_Time_min        1000 non-null   int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 78.3+ KB


In [79]:

df.describe()

,Order_ID,Distance_km,Order_Time,Restaurant_Rating,Prep_Time_min,Delivery_Partner_Rating,Delivery_Time_min
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1500.500000,5.157360,16.15000,3.988600,24.892000,4.248100,66.614000
std,288.819436,2.775165,3.81314,0.576196,8.405771,0.431048,18.021537
min,1001.000000,0.540000,10.00000,3.000000,10.000000,3.500000,21.000000
25%,1250.750000,2.740000,13.00000,3.500000,18.000000,3.900000,53.000000
50%,1500.500000,5.220000,16.00000,4.000000,25.000000,4.200000,67.000000
75%,1750.250000,7.572500,19.00000,4.500000,32.000000,4.600000,80.000000
max,2000.000000,10.000000,22.00000,5.000000,39.000000,5.000000,119.000000


In [80]:
print(df.isnull().sum())

Order_ID                   0
Distance_km                0
Order_Time                 0
Traffic_Level              0
Weather                    0
Restaurant_Rating          0
Prep_Time_min              0
Delivery_Partner_Rating    0
Vehicle_Type               0
Delivery_Time_min          0
dtype: int64


### 4. Data Preprocessing

#### 4.1 Drop Irrelevant Column

In [81]:
df.drop('Order_ID', axis=1, inplace=True)

#### 4.2 Feature Engineering

In [82]:
df['Distance_x_Traffic'] = df['Distance_km'] * df['Traffic_Level'].map({'Low': 1, 'Medium': 2, 'High': 3})
df['Rating_Diff']        = df['Restaurant_Rating'] - df['Delivery_Partner_Rating']
df['Total_Time_Est']     = df['Distance_km'] * 5 + df['Prep_Time_min']
df['Rush_Hour']          = df['Order_Time'].apply(lambda x: 1 if x in range(18, 22) else 0)

#### 4.3 Encoding

In [83]:
df = pd.get_dummies(df, drop_first=True, dtype=int)

### 5. Model Building

#### 5.1 Split Features & Target

In [84]:
X = df.drop('Delivery_Time_min', axis=1)
y = df['Delivery_Time_min']

#### 5.2 Train-Test Split

In [85]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#### 5.3. Model Before Tuning

- #### Model Selection

In [86]:
model_before = AdaBoostRegressor(random_state=42)

- #### Train Model

In [87]:
model_before.fit(X_train, y_train)

AdaBoostRegressor(random_state=42)

- #### Make Predictions

In [88]:

y_pred_before = model_before.predict(X_test)

- #### Model Evaluation

In [89]:
print("BEFORE TUNING")
print("MAE  :", mean_absolute_error(y_test, y_pred_before))
print("MSE  :", mean_squared_error(y_test, y_pred_before))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_before)))
print("R²   :", r2_score(y_test, y_pred_before))

BEFORE TUNING
MAE  : 4.836371076722753
MSE  : 35.917199935627565
RMSE : 5.993096022560256
R²   : 0.9029138326593309


#### 5.4. Hyperparameter Tuning

- #### Define Hyperparameters

In [90]:
params = {
    'n_estimators' : [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0],
    'loss'         : ['linear', 'square', 'exponential'],
    'estimator'    : [
        DecisionTreeRegressor(max_depth=3),
        DecisionTreeRegressor(max_depth=5),
        DecisionTreeRegressor(max_depth=7)
    ]
}

- ####  Apply GridSearchCV

In [91]:
grid = GridSearchCV(
    AdaBoostRegressor(random_state=42),
    param_grid=params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

#### 5.5 Model After Tuning

- #### Train with GridSearchCV

In [92]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=AdaBoostRegressor(random_state=42), n_jobs=-1,
             param_grid={'estimator': [DecisionTreeRegressor(max_depth=3),
                                       DecisionTreeRegressor(max_depth=5),
                                       DecisionTreeRegressor(max_depth=7)],
                         'learning_rate': [0.01, 0.1, 0.5, 1.0],
                         'loss': ['linear', 'square', 'exponential'],
                         'n_estimators': [50, 100, 200]},
             scoring='r2')

- #### Build Best Model

In [93]:
best_model = grid.best_estimator_

- #### Make Predictions

In [94]:
y_pred_after = best_model.predict(X_test)

- #### Model Evaluation

In [95]:
print("AFTER TUNING")
print("Best Parameters :", grid.best_params_)
print("MAE  :", mean_absolute_error(y_test, y_pred_after))
print("MSE  :", mean_squared_error(y_test, y_pred_after))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_after)))
print("R²   :", r2_score(y_test, y_pred_after))

AFTER TUNING
Best Parameters : {'estimator': DecisionTreeRegressor(max_depth=7), 'learning_rate': 0.5, 'loss': 'square', 'n_estimators': 200}
MAE  : 3.6658624591759352
MSE  : 19.830967837489112
RMSE : 4.4531974846720095
R²   : 0.9463958029732683


#### 5.6 Prediction Report

In [96]:
print(pd.DataFrame({
    'Actual'   : y_test.values,
    'Predicted': y_pred_after.round(2),
    'Error'    : (y_test.values - y_pred_after).round(2)
}).head(10))

   Actual  Predicted  Error
0      73      74.00  -1.00
1     100     106.00  -6.00
2      75      79.00  -4.00
3      79      71.18   7.82
4      86      85.00   1.00
5     109     101.25   7.75
6      70      71.93  -1.93
7      94      92.34   1.66
8      44      47.50  -3.50
9      74      76.08  -2.08


### **Key Insights**
- R² improved from 90.29% → 94.64% after hyperparameter tuning 

- All 4 metrics improved significantly after tuning 
- MAE reduced from 4.836 → 3.666 (better prediction accuracy) 
- MSE reduced from 35.917 → 19.831 (massive reduction!) 
- RMSE reduced from 5.993 → 4.453 
- R² improvement of +4.35% — outstanding for boosting model 
- Best Parameters: estimator=DecisionTreeRegressor(max_depth=7),
  learning_rate=0.5, loss=square, n_estimators=200
- max_depth=7 → deeper weak learners captured complex patterns better
- loss=square → penalizes larger errors more — better for regression 
- Feature Engineering added 4 meaningful features:
  Distance_x_Traffic, Rating_Diff, Total_Time_Est, Rush_Hour
- Average prediction error = only 3.67 minutes — production ready! 
- AdaBoost Regressor outperformed baseline by a large margin

### **Conclusion**
The AdaBoost Regressor successfully predicted food delivery time with
an outstanding R² Score of 94.64% after hyperparameter tuning —
a significant improvement from the 90.29% baseline (+4.35%).

Feature Engineering added powerful interaction features like
Distance_x_Traffic and Total_Time_Est that helped the model capture
real-world delivery patterns. GridSearchCV found the optimal
combination of n_estimators=200, learning_rate=0.5, loss=square
and base estimator with max_depth=7.

With an average prediction error of only 3.67 minutes, this model
is production-ready for real-world food delivery platforms. This
project demonstrates the strength of sequential boosting ensemble
methods for regression tasks in the food delivery domain.